# Luvansh + IGI Diamond Filter (Colab, single notebook)

This notebook:
1. Scrapes **10 diamonds at a time** from Luvansh in the 1.8-2.5ct range.
2. Opens each product page and extracts the IGI report URL.
3. Opens IGI, finds the **PDF Report**, downloads the PDF, and extracts grading text.
4. Filters stones by your target geometry ranges.


In [ ]:
# Install deps in Colab
!pip -q install requests beautifulsoup4 pandas lxml pypdf


In [ ]:
import io
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pypdf import PdfReader

BASE = 'https://www.luvansh.com'
SHOP_URL = f'{BASE}/shop-diamond'

TARGET = {
    'ratio_min': 1.00,
    'ratio_max': 1.02,
    'table_min': 54.0,
    'table_max': 58.0,
    'depth_min': 61.0,
    'depth_max': 62.3,
    'crown_angle_min': 34.0,
    'crown_angle_max': 35.0,
    'pavilion_angle_min': 40.6,
    'pavilion_angle_max': 40.9,
    'crown_height_min': 14.0,
    'crown_height_max': 16.0,
    'pavilion_depth_min': 42.5,
    'pavilion_depth_max': 43.2,
}


In [ ]:
def get_soup(url, params=None):
    headers = {
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122 Safari/537.36'
    }
    r = requests.get(url, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, 'lxml')

def parse_float(pattern, text):
    m = re.search(pattern, text, flags=re.IGNORECASE)
    return float(m.group(1)) if m else None

def find_pdf_url_from_igi_html(html, base_url):
    soup = BeautifulSoup(html, 'lxml')
    for a in soup.select('a[href]'):
        href = a.get('href', '')
        if '.pdf' in href.lower():
            return urljoin(base_url, href)
    m = re.search(r'https?://[^"\s>]+\.pdf', html, flags=re.IGNORECASE)
    return m.group(0) if m else None

def extract_igi_metrics_from_pdf_bytes(pdf_bytes):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    text = '\n'.join((p.extract_text() or '') for p in reader.pages)
    data = {
        'raw_text': text,
        'table_pct': parse_float(r'Table\s*(\d+(?:\.\d+)?)%', text),
        'depth_pct': parse_float(r'Depth\s*(\d+(?:\.\d+)?)%', text),
        'crown_angle': parse_float(r'(\d+(?:\.\d+)?)°\s*(?:Crown|CROWN)', text),
        'pavilion_angle': parse_float(r'(\d+(?:\.\d+)?)°\s*(?:Pavilion|PAVILION)', text),
        'crown_height_pct': parse_float(r'Crown\s*Height\s*(\d+(?:\.\d+)?)%', text),
        'pavilion_depth_pct': parse_float(r'Pavilion\s*Depth\s*(\d+(?:\.\d+)?)%', text),
    }
    if data['crown_angle'] is None or data['pavilion_angle'] is None:
        deg_vals = [float(v) for v in re.findall(r'(\d+(?:\.\d+)?)°', text)]
        if len(deg_vals) >= 2:
            data['crown_angle'] = data['crown_angle'] or deg_vals[0]
            data['pavilion_angle'] = data['pavilion_angle'] or deg_vals[1]
    if data['crown_height_pct'] is None or data['pavilion_depth_pct'] is None:
        pct_vals = [float(v) for v in re.findall(r'(\d+(?:\.\d+)?)%', text)]
        if len(pct_vals) >= 5:
            data['crown_height_pct'] = data['crown_height_pct'] or pct_vals[0]
            data['pavilion_depth_pct'] = data['pavilion_depth_pct'] or pct_vals[4]
    return data

def extract_ratio_from_measurements(measurements_text):
    nums = [float(x) for x in re.findall(r'\d+(?:\.\d+)?', measurements_text)]
    if len(nums) >= 2:
        a, b = nums[0], nums[1]
        return round(max(a, b) / min(a, b), 4)
    return None


In [ ]:
def collect_luvansh_products_batch(carat_min=1.8, carat_max=2.5, batch_size=10, page=1):
    params = {'page': page, 'carat': f'{carat_min},{carat_max}'}
    soup = get_soup(SHOP_URL, params=params)
    links = []
    for a in soup.select('a[href*="/product/"]'):
        href = a.get('href')
        if href:
            full = urljoin(BASE, href)
            if '/product/' in full and full not in links:
                links.append(full)
    return links[:batch_size]

def get_igi_link_from_product(product_url):
    soup = get_soup(product_url)
    for a in soup.select('a[href]'):
        href = a.get('href', '')
        if 'igi.org/Verify-Your-Report' in href:
            return href
    txt = soup.get_text(' ', strip=True)
    m = re.search(r'https://www\.igi\.org/Verify-Your-Report/\?r=\w+', txt)
    return m.group(0) if m else None

def metric_ok(v, lo, hi):
    return (v is not None) and (lo <= v <= hi)

def is_match(row):
    return all([
        metric_ok(row.get('ratio'), TARGET['ratio_min'], TARGET['ratio_max']),
        metric_ok(row.get('table_pct'), TARGET['table_min'], TARGET['table_max']),
        metric_ok(row.get('depth_pct'), TARGET['depth_min'], TARGET['depth_max']),
        metric_ok(row.get('crown_angle'), TARGET['crown_angle_min'], TARGET['crown_angle_max']),
        metric_ok(row.get('pavilion_angle'), TARGET['pavilion_angle_min'], TARGET['pavilion_angle_max']),
        metric_ok(row.get('crown_height_pct'), TARGET['crown_height_min'], TARGET['crown_height_max']),
        metric_ok(row.get('pavilion_depth_pct'), TARGET['pavilion_depth_min'], TARGET['pavilion_depth_max']),
    ])


In [ ]:
batch_links = collect_luvansh_products_batch(carat_min=1.8, carat_max=2.5, batch_size=10, page=1)
print(f'Found {len(batch_links)} product links in this batch')

rows = []
for product_url in batch_links:
    row = {'product_url': product_url, 'igi_verify_url': None, 'igi_pdf_url': None, 'ratio': None, 'table_pct': None, 'depth_pct': None, 'crown_angle': None, 'pavilion_angle': None, 'crown_height_pct': None, 'pavilion_depth_pct': None, 'match': False, 'error': None}
    try:
        soup = get_soup(product_url)
        text = soup.get_text(' ', strip=True)
        mm_match = re.search(r'(\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?\s*X\s*\d+(?:\.\d+)?\s*MM)', text, flags=re.IGNORECASE)
        if mm_match:
            row['ratio'] = extract_ratio_from_measurements(mm_match.group(1))
        igi_verify = get_igi_link_from_product(product_url)
        row['igi_verify_url'] = igi_verify
        if igi_verify:
            igi_resp = requests.get(igi_verify, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
            igi_resp.raise_for_status()
            pdf_url = find_pdf_url_from_igi_html(igi_resp.text, igi_verify)
            row['igi_pdf_url'] = pdf_url
            if pdf_url:
                pdf_resp = requests.get(pdf_url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
                pdf_resp.raise_for_status()
                metrics = extract_igi_metrics_from_pdf_bytes(pdf_resp.content)
                for k in ['table_pct', 'depth_pct', 'crown_angle', 'pavilion_angle', 'crown_height_pct', 'pavilion_depth_pct']:
                    row[k] = metrics.get(k)
        row['match'] = is_match(row)
    except Exception as e:
        row['error'] = str(e)
    rows.append(row)

df = pd.DataFrame(rows)
display(df)
matches = df[df['match'] == True].copy()
print(f'\nMatches found: {len(matches)}')
display(matches)
df.to_csv('diamonds_batch_all.csv', index=False)
matches.to_csv('diamonds_batch_matches.csv', index=False)
print('Saved: diamonds_batch_all.csv, diamonds_batch_matches.csv')


## Notes
- If Luvansh/IGI blocks requests, run the same logic with Selenium in Colab.
- To fetch the next 10 diamonds, rerun with `page=2` (then `page=3`, etc.).
- You can tighten/loosen ranges by editing the `TARGET` dictionary.
